# Lab22 bonus — Kaggle · Nguyễn Vũ Anh · 2A202602502

Chạy tiếp từ checkpoint core đã lưu; chưa có kết quả bonus trong bản chuẩn bị này.
Bật Internet và GPU T4 x2; code dùng GPU thứ nhất, không cộng VRAM hai GPU.
Gắn dataset riêng chứa `Lab22_checkpoint_100.zip` vào Input trước khi chạy.
Chạy từng mục, lưu phiên bản có output và tải ZIP bằng chứng sau từng mục.
Giữ ngân sách T4 gốc: variants300, beta800train/100eval, GRPO60steps/100test,
IFEval200/GSM8K250/MMLU10 mỗi môn. API judge/HF Hub cần cấu hình riêng.


In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['COMPUTE_TIER'] = 'T4'
os.environ['HF_HOME'] = '/kaggle/temp/hf-cache'
os.environ['MPLBACKEND'] = 'Agg'
import torch
assert torch.cuda.is_available(), 'Bật GPU T4 trong Session options trước khi chạy.'
WORK = Path('/kaggle/working/lab22-bonus')
if not (WORK/'.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/vuanh259/K4-Track3-Day22-NguyenVuAnh-2A202602502-DPO-ORPO-Alignment', str(WORK)], check=True)
os.chdir(WORK)
print(torch.cuda.get_device_name(), Path.cwd())


In [ ]:
%pip install -q "unsloth>=2026.10.1" "trl>=1.13,<1.14" "transformers>=5.2,<5.18" "peft>=0.18,<1.0" "accelerate>=1.10,<2.0" "bitsandbytes>=0.48,<1.0" "datasets>=4.7,<5.0" "matplotlib>=3.9,<4.0" "pandas>=2.2,<4.0" "pyarrow>=17" "lm-eval[ifeval,math]>=0.4.13,<0.5"

In [ ]:
import hashlib
def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
candidates = list(Path('/kaggle/input').rglob('Lab22_checkpoint_100.bin'))
if not candidates:
    candidates = list(Path('/kaggle/input').rglob('Lab22_checkpoint_100.zip'))
if candidates:
    assert len(candidates)==1
    checkpoint = candidates[0]
    assert digest(checkpoint)=='28dd3541885fbac9ad23327c5255f4895367d18d8b2c847262e07455dc3064d9'
else:
    sfts = [p for p in Path('/kaggle/input').rglob('adapter_model.safetensors') if p.parent.name=='sft-mini']
    assert len(sfts)==1, 'Gắn dataset checkpoint trước khi chạy.'
    checkpoint = sfts[0].parents[2]
    assert digest(sfts[0])=='1a482bfa68e7aad3d03ba5597bfba88dadc2de1f1706cf94913fd2363a06ffb4'
    assert digest(checkpoint/'adapters/dpo-checkpoints/checkpoint-100/adapter_model.safetensors')=='cd0219741bd7a37317ed3adde86a9cfa8ba0cfe8f90cf5e767ac1c8307e33f8e'
print('Verified saved checkpoint:', checkpoint)


In [ ]:
(WORK/'scripts/restore_lab22_checkpoint.py').write_text('"""Restore the saved core adapters and rebuild the original SFT reference.\n\nUsage: python scripts/restore_lab22_checkpoint.py /path/Lab22_checkpoint_100.zip\nRequires the lab CUDA/Unsloth environment. Does not retrain either adapter.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport sys\nimport zipfile\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef extract_checkpoint(archive: Path, destination: Path) -> None:\n    with zipfile.ZipFile(archive) as z:\n        for member in z.infolist():\n            target = (destination / member.filename).resolve()\n            if not target.is_relative_to(destination.resolve()):\n                raise ValueError(f"Unsafe archive member: {member.filename}")\n        z.extractall(destination)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("archive", type=Path)\n    args = parser.parse_args()\n    sys.path.insert(0, str(ROOT))\n    import unsloth  # must patch before transformers/peft\n    import torch\n    from lab22 import config as C\n    from lab22 import modeling as MD\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU required; no training or restore has run.")\n    if args.archive.is_dir():\n        for rel in ("adapters/sft-mini", "adapters/dpo-checkpoints/checkpoint-100"):\n            source = args.archive / rel\n            if not source.is_dir():\n                raise FileNotFoundError(source)\n            shutil.copytree(source, ROOT / rel, dirs_exist_ok=True)\n    else:\n        extract_checkpoint(args.archive, ROOT)\n    checkpoint = C.ADAPTERS / "dpo-checkpoints" / "checkpoint-100"\n    if not (checkpoint / "adapter_model.safetensors").is_file():\n        raise FileNotFoundError("Missing saved DPO weights")\n    if not (C.SFT_ADAPTER / "adapter_model.safetensors").is_file():\n        raise FileNotFoundError("Missing saved SFT weights")\n    model, tokenizer = MD.load_model(C.SFT_ADAPTER)\n    model.save_pretrained_merged(str(C.SFT_MERGED), tokenizer, save_method="merged_16bit")\n    del model\n    MD.cleanup()\n    C.DPO_ADAPTER.mkdir(parents=True, exist_ok=True)\n    for name in ("adapter_model.safetensors", "adapter_config.json", "trainer_state.json"):\n        shutil.copy2(checkpoint / name, C.DPO_ADAPTER / name)\n    cfg_path = C.DPO_ADAPTER / "adapter_config.json"\n    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))\n    cfg["base_model_name_or_path"] = str(C.SFT_MERGED.resolve())\n    cfg_path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")\n    tokenizer.save_pretrained(str(C.DPO_ADAPTER))\n    for folder, filename in ((C.SFT_ADAPTER, "adapter_model.safetensors"),\n                             (C.DPO_ADAPTER, "adapter_model.safetensors")):\n        p = folder / filename\n        print(p.relative_to(ROOT), hashlib.sha256(p.read_bytes()).hexdigest())\n    print("Restored saved adapters; rebuilt SFT reference. No retraining.")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
subprocess.run([sys.executable, 'scripts/restore_lab22_checkpoint.py', str(checkpoint)], check=True)


In [ ]:
def backup_bonus(label):
    import zipfile
    from IPython.display import display, FileLink
    target = Path('/kaggle/working')/f'Lab22_bonus_{label}.zip'
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as z:
        for parent in ('adapters', 'data/eval', 'submission'):
            for p in (WORK/parent).rglob('*'):
                if p.is_file() and p.suffix in ('.json', '.jsonl', '.png', '.md', '.txt'):
                    z.write(p, p.relative_to(WORK).as_posix())
    print('Evidence backup:', target, target.stat().st_size)
    display(FileLink(str(target)))


In [ ]:
# Free GPU memory held by the previous stage (model, trainer, llama.cpp handle).
import gc
for _name in ('trainer', 'model', 'ref_model', 'llm', 'policy', 'ref', 'tokenizer'):
    globals().pop(_name, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
except ImportError:
    pass

# NB3b — So sánh các biến thể: DPO · RPO · DPO chuẩn hoá độ dài · LD-DPO · ORPO (TUỲ CHỌN, +8)

Cùng dữ liệu (`VARIANT_TRAIN` cặp đầu của NB2), cùng số bước, cùng LoRA. Chỉ đổi loss.

| Run | Cấu hình TRL | Ý tưởng |
|---|---|---|
| `dpo` | `loss_type=["sigmoid"]` | mức cơ sở (baseline) |
| `rpo` | `loss_type=["sigmoid","sft"]` | thêm NLL trên chosen, chống likelihood displacement |
| `dpo_norm` | `loss_type=["sigmoid_norm"]` | log-prob trung bình theo token (gần SimPO nhưng vẫn có reference) |
| `ld_dpo` | `ld_alpha=0.5` | giảm trọng số phần token vượt quá độ dài chung (LD-DPO) |
| `orpo` | `trl.experimental.orpo` | không reference, SFT + odds-ratio trong một bước |

ORPO thường xuất phát từ mô hình *chưa* SFT; ở đây nó chạy trên cùng `models/sft-merged`
để bảng so sánh chỉ khác nhau ở loss. Đặt `ORPO_FROM_BASE=1` để thử ORPO từ base.

**Đọc kết quả:** độ chính xác reward trên held-out và độ dài đầu ra trung bình. Các
biến thể có thang reward khác nhau, nên **không so trực tiếp giá trị margin** giữa các dòng.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab22" / "config.py").exists())
sys.path.insert(0, str(ROOT))

import unsloth  # noqa: F401
import torch

from lab22 import config as C
from lab22 import modeling as MD

assert torch.cuda.is_available()
assert C.SFT_MERGED.exists() and (C.PREF_DIR / "train.parquet").exists(), "Run NB1 + NB2 first"

from datasets import Dataset

train_ds = Dataset.from_parquet(str(C.PREF_DIR / "train.parquet"))
train_ds = train_ds.select(range(min(C.TIER.variant_train, len(train_ds))))
assert len(train_ds) > 0, "Empty preference train split: rerun NB2"
eval_ds = Dataset.from_parquet(str(C.PREF_DIR / "eval.parquet"))
probe_prompts = [r["prompt"][0]["content"] for r in eval_ds.select(range(min(20, len(eval_ds))))]
print(f"variant train={len(train_ds)} eval={len(eval_ds)} probe prompts={len(probe_prompts)}")

RUNS = {
    "dpo": {"loss_type": ["sigmoid"]},
    "rpo": {"loss_type": ["sigmoid", "sft"], "loss_weights": [1.0, 1.0]},
    "dpo_norm": {"loss_type": ["sigmoid_norm"]},
    "ld_dpo": {"loss_type": ["sigmoid"], "ld_alpha": 0.5},
}
SELECTED = [r for r in os.environ.get("VARIANTS", "dpo,rpo,dpo_norm,ld_dpo,orpo").split(",") if r]

## 1. Các biến thể dựa trên DPOTrainer

In [ ]:
import json


from trl import DPOTrainer


summary_path = C.VARIANTS_DIR / "variants_summary.json"
results = json.loads(summary_path.read_text()) if summary_path.exists() else {}
for name in [r for r in SELECTED if r in RUNS]:
    if name in results:
        print("Already completed:", name)
        continue
    print(f"\n=== {name} ===")
    model, tokenizer = MD.load_model(C.SFT_MERGED)
    model = MD.add_lora(model)
    overrides = dict(RUNS[name])
    loss_type = overrides.pop("loss_type")
    trainer = DPOTrainer(
        model=model,
        args=MD.dpo_config(C.VARIANTS_DIR / f"{name}-ckpt", loss_type=loss_type, eval_strategy="no", **overrides),
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
    )
    trainer.train()
    ev = trainer.evaluate()
    outputs = MD.generate(trainer.model, tokenizer, probe_prompts, max_new_tokens=256)
    results[name] = {
        "eval_reward_accuracy": ev.get("eval_rewards/accuracies"),
        "eval_chosen_reward": ev.get("eval_rewards/chosen"),
        "eval_rejected_reward": ev.get("eval_rewards/rejected"),
        "mean_output_chars": sum(map(len, outputs)) / len(outputs),
        "diagnosis": MD.diagnose(MD.reward_history(trainer.state.log_history))[0],
    }
    trainer.model.save_pretrained(str(C.VARIANTS_DIR / name))
    (C.VARIANTS_DIR / name / "run_evidence.json").write_text(json.dumps({"metrics": results[name], "eval": ev, "history": trainer.state.log_history, "prompts": probe_prompts, "outputs": outputs, "train_samples": len(train_ds), "eval_samples": len(eval_ds)}, indent=2, ensure_ascii=False, default=str))
    summary_path.write_text(json.dumps(results, indent=2, ensure_ascii=False, default=str))
    backup_bonus("variant_" + name)
    print(results[name])
    del trainer, model
    MD.cleanup()

## 2. ORPO (không reference)

In [ ]:
if "orpo" in SELECTED:
    from trl.experimental.orpo import ORPOConfig, ORPOTrainer

    start = C.BASE_MODEL if os.environ.get("ORPO_FROM_BASE") == "1" else C.SFT_MERGED
    model, tokenizer = MD.load_model(start)
    model = MD.add_lora(model)
    orpo_args = ORPOConfig(
        output_dir=str(C.VARIANTS_DIR / "orpo-ckpt"),
        per_device_train_batch_size=C.TIER.dpo_batch,
        per_device_eval_batch_size=C.TIER.dpo_batch,
        gradient_accumulation_steps=C.TIER.dpo_grad_accum,
        num_train_epochs=C.DPO_EPOCHS,
        learning_rate=C.DPO_LR,
        beta=0.1,  # λ in the paper
        max_length=C.MAX_LEN,
        warmup_steps=0.1,
        lr_scheduler_type="cosine",
        logging_steps=5,
        save_strategy="no",
        optim="adamw_8bit",
        seed=C.SEED,
        report_to="none",
        **MD.precision_flags(),
    )
    trainer = ORPOTrainer(
        model=model, args=orpo_args, train_dataset=train_ds, eval_dataset=eval_ds, processing_class=tokenizer
    )
    trainer.train()
    ev = trainer.evaluate()
    outputs = MD.generate(trainer.model, tokenizer, probe_prompts, max_new_tokens=256)
    results["orpo"] = {
        "start": str(start),
        "eval_reward_accuracy": ev.get("eval_rewards/accuracies"),
        "eval_log_odds_ratio": ev.get("eval_log_odds_ratio"),
        "mean_output_chars": sum(map(len, outputs)) / len(outputs),
    }
    trainer.model.save_pretrained(str(C.VARIANTS_DIR / "orpo"))
    print(results["orpo"])
    del trainer, model
    MD.cleanup()

## 3. Bảng tổng hợp (sản phẩm nộp `03b-variants.png`)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

table = pd.DataFrame(results).T
print(table.to_string())
(C.VARIANTS_DIR / "variants_summary.json").write_text(json.dumps(results, indent=2, default=str))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
table["eval_reward_accuracy"].astype(float).plot.bar(ax=axes[0], color="#2e548a")
axes[0].set_ylim(0, 1)
axes[0].set_title("held-out reward accuracy")
table["mean_output_chars"].astype(float).plot.bar(ax=axes[1], color="#c83538")
axes[1].set_title("mean output length (chars)")
fig.tight_layout()
fig.savefig(C.SCREENSHOTS / "03b-variants.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Câu hỏi cho REFLECTION

1. Biến thể nào làm đầu ra dài ra nhiều nhất? Có khớp với tỉ lệ "chosen dài hơn" ở NB2 không?
2. RPO có giữ `rewards/chosen` dương trong khi DPO thì không? (so với chẩn đoán NB3)
3. Độ chính xác reward cao hơn có nghĩa là mô hình tốt hơn không? Chạy giám khảo ở NB4 trên
   adapter `adapters/variants/<name>` (đặt `DPO_ADAPTER_OVERRIDE`) để kiểm tra.

In [ ]:
backup_bonus('03b_dpo_variants')

In [ ]:
# Free GPU memory held by the previous stage (model, trainer, llama.cpp handle).
import gc
for _name in ('trainer', 'model', 'ref_model', 'llm', 'policy', 'ref', 'tokenizer'):
    globals().pop(_name, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
except ImportError:
    pass

In [ ]:
%pip install -q "llama-cpp-python>=0.3.16,<1.0"

# NB5 — Gộp SFT+DPO → GGUF Q4_K_M (TUỲ CHỌN, thưởng điểm)

> Phần lõi của lab = NB0–NB4. Bước này biên dịch llama.cpp lúc chạy (~3–5 phút lần đầu).

**Lỗi của lab cũ:** NB5 chỉ nạp adapter **SFT** rồi gộp, nên file GGUF không có DPO.
Ở đây ta nạp `adapters/dpo` (adapter config trỏ tới `models/sft-merged`), tức là
SFT + DPO, rồi xuất file. Có một bước kiểm tra để chắc chắn adapter DPO thực sự được nạp.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab22" / "config.py").exists())
sys.path.insert(0, str(ROOT))

import unsloth  # noqa: F401
import torch

from lab22 import config as C
from lab22 import modeling as MD

assert torch.cuda.is_available()
assert (C.DPO_ADAPTER / "adapter_config.json").exists(), "Run NB3 first"
C.ensure_dirs()

adapter_cfg = json.loads((C.DPO_ADAPTER / "adapter_config.json").read_text())
print(f"DPO adapter base: {adapter_cfg.get('base_model_name_or_path')}")

## 1. Nạp SFT + DPO ở 16-bit

Gộp vào trọng số 4-bit làm mất độ chính xác, nên nạp 16-bit (`load_in_4bit=False`).
T4 16 GB đủ cho 4B ở fp16 (~8 GB).

In [ ]:
model, tokenizer = MD.load_model(C.DPO_ADAPTER, load_in_4bit=False)
n_lora = sum(1 for n, _ in model.named_parameters() if "lora_" in n)
assert n_lora > 0, "DPO LoRA weights not loaded: the GGUF would be SFT-only"
print(f"LoRA tensors loaded: {n_lora}")

SMOKE_PROMPT = "Giải thích ngắn gọn (3 câu) cách thuật toán Bubble sort hoạt động."
SMOKE_TOKENS = 160
hf_answer = MD.generate(model, tokenizer, [SMOKE_PROMPT], max_new_tokens=SMOKE_TOKENS)[0]
# Render the prompt once with the HF template (thinking off) and feed GGUF the same text,
# so the HF vs GGUF comparison differs only by quantization, not by chat formatting.
smoke_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": SMOKE_PROMPT}], tokenize=False, add_generation_prompt=True, **C.CHAT_TEMPLATE_KWARGS
)
stop_tokens = [tokenizer.eos_token] if tokenizer.eos_token else []
print(f"HF (SFT+DPO) answer:\n{hf_answer}")

## 2. Xuất GGUF Q4_K_M

`save_pretrained_gguf` gộp LoRA vào trọng số 16-bit rồi gọi llama.cpp để lượng tử hoá.
Tên file và thư mục con khác nhau giữa các bản Unsloth, nên ta tìm file bằng glob đệ quy.

In [ ]:
import shutil
scratch_gguf = Path("/kaggle/temp/lab22-gguf-export")
scratch_gguf.mkdir(parents=True, exist_ok=True)
print("Free output GB:", shutil.disk_usage(C.REPO_ROOT).free / 1e9)
print("Free scratch GB:", shutil.disk_usage(scratch_gguf).free / 1e9)
model.save_pretrained_gguf(str(scratch_gguf), tokenizer, quantization_method="q4_k_m")
hits = [p for p in scratch_gguf.rglob("*.gguf") if "q4_k_m" in p.name.lower()]
assert hits, "Q4_K_M file missing after export"
source_gguf = max(hits, key=lambda p:p.stat().st_mtime)
C.GGUF_DIR.mkdir(parents=True, exist_ok=True)
gguf_path = C.GGUF_DIR / source_gguf.name
shutil.copy2(source_gguf, gguf_path)
assert gguf_path.stat().st_size == source_gguf.stat().st_size
print(f"{gguf_path.relative_to(C.REPO_ROOT)}  {gguf_path.stat().st_size / 1e9:.2f} GB")
del model
MD.cleanup()


## 3. Kiểm tra nhanh (smoke test) bằng llama-cpp-python (sản phẩm nộp `06-gguf-smoke.png`)

So câu trả lời GGUF với câu trả lời HF ở §1. Lượng tử hoá 4-bit làm câu chữ lệch một
chút, nhưng nội dung và phong cách phải giống nhau. Nếu GGUF trả lời giống hệt
mô hình SFT (NB4) thì adapter DPO đã bị bỏ sót.

In [ ]:
from llama_cpp import Llama

llm = Llama(model_path=str(gguf_path), n_ctx=C.MAX_LEN, n_gpu_layers=-1, verbose=False)
resp = llm.create_completion(smoke_text, max_tokens=SMOKE_TOKENS, temperature=0.0, stop=stop_tokens)
gguf_answer = resp["choices"][0]["text"].strip()
print(f"PROMPT: {SMOKE_PROMPT}\n\nGGUF Q4_K_M:\n{gguf_answer}\n\nusage: {resp['usage']}")
llm.close()  # release the llama.cpp GPU buffers before the next notebook
del llm

In [ ]:
deploy_meta = {
    "compute_tier": C.COMPUTE_TIER,
    "base_model": C.BASE_MODEL,
    "adapter": str(C.DPO_ADAPTER.relative_to(C.REPO_ROOT)),
    "adapter_base": adapter_cfg.get("base_model_name_or_path"),
    "gguf_path": str(gguf_path.relative_to(C.REPO_ROOT)),
    "gguf_size_mb": round(gguf_path.stat().st_size / 1e6, 1),
    "quantization": "q4_k_m",
    "smoke_prompt": SMOKE_PROMPT,
    "chat_template_kwargs": C.CHAT_TEMPLATE_KWARGS,
    "max_new_tokens": SMOKE_TOKENS,
    "hf_answer": hf_answer,
    "gguf_answer": gguf_answer,
}
(C.EVAL_DIR / "deploy_meta.json").write_text(json.dumps(deploy_meta, ensure_ascii=False, indent=2))
print("Saved data/eval/deploy_meta.json")
import textwrap
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, title, answer in zip(axes, ["HF fp16: SFT + DPO", "GGUF Q4_K_M: SFT + DPO"], [hf_answer, gguf_answer]):
    ax.axis("off")
    ax.set_title(title)
    ax.text(0, 1, textwrap.fill(answer, width=70), transform=ax.transAxes, va="top", fontsize=10)
fig.suptitle(textwrap.fill(SMOKE_PROMPT, width=100), fontsize=12)
fig.tight_layout(rect=[0,0,1,0.94])
fig.savefig(C.SCREENSHOTS / "06-gguf-smoke.png", dpi=120, bbox_inches="tight")
plt.show()


## 4. Triển khai (tham khảo, chạy ngoài notebook)

Máy chủ llama.cpp (CPU/GPU, file GGUF):

```bash
llama-server -m gguf/<file>.Q4_K_M.gguf --port 8080 -c 2048
```

vLLM (BigGPU, ≥16 GB): phục vụ mô hình 16-bit có LoRA mà không cần gộp:

```bash
vllm serve models/sft-merged --enable-lora --lora-modules dpo=adapters/dpo \
  --max-model-len 2048 --port 8000
```

Gọi API với `"model": "dpo"`. vLLM giữ tiến trình và GPU, nên chạy ở cửa sổ dòng lệnh riêng.

**Tiếp theo:** NB6 (benchmark) hoặc `make verify`.

In [ ]:
backup_bonus('05_merge_deploy_gguf')

In [ ]:
# Free GPU memory held by the previous stage (model, trainer, llama.cpp handle).
import gc
for _name in ('trainer', 'model', 'ref_model', 'llm', 'policy', 'ref', 'tokenizer'):
    globals().pop(_name, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
except ImportError:
    pass

# NB6 — Đánh giá chuẩn (benchmark) SFT vs SFT+DPO bằng lm-eval (TUỲ CHỌN, thưởng điểm)

**Công nghệ:** `lm-eval` 0.4.13 trên mô hình 16-bit: `models/sft-merged` (SFT) và
`models/sft-merged` + `peft=adapters/dpo` (SFT+DPO).

Ba lỗi của bản cũ đã sửa:
1. **Chat template.** Mô hình chat bị chấm như mô hình base (không template) thì điểm
   IFEval/GSM8K không phản ánh cách nó được dùng. Ở đây truyền `--apply_chat_template`
   và `--fewshot_as_multiturn`.
2. **`--limit` tính theo từng subtask.** `--limit 500` trên nhóm `mmlu` (57 môn) là
   ~28k câu, không phải 500. Limit MMLU ở đây được đặt *theo môn*.
3. **AlpacaEval-lite bị bỏ.** `tatsu-lab/alpaca_eval` là bộ dữ liệu dạng script, không
   nạp được với `datasets` ≥ 4. Đánh giá theo kiểu giám khảo đã có ở NB4.

Thêm `global_mmlu_full_vi` (MMLU dịch tiếng Việt, có kiểm duyệt) vì lab là tiếng Việt.

> Điểm có thể **giảm** sau DPO (alignment tax, tức thuế căn chỉnh). Với limit nhỏ, chênh lệch ±2–3 điểm
> thường nằm trong nhiễu: xem cột `stderr` trước khi kết luận.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab22" / "config.py").exists())
sys.path.insert(0, str(ROOT))

import torch

from lab22 import config as C

assert torch.cuda.is_available()
assert C.SFT_MERGED.exists() and C.DPO_ADAPTER.exists(), "Run NB1 + NB3 first"
C.ensure_dirs()

BIG = C.COMPUTE_TIER == "BIGGPU"
# name: (task, num_fewshot, limit per subtask or None for all, primary metric)
BENCHMARKS = {
    "IFEval": ("ifeval", 0, None if BIG else 200, "prompt_level_strict_acc,none"),
    "GSM8K": ("gsm8k", 5, None if BIG else 250, "exact_match,flexible-extract"),
    "Global-MMLU-vi": ("global_mmlu_full_vi", 0, 40 if BIG else 10, "acc,none"),
}
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
BATCH = "auto" if BIG else "4"
for name, (task, shots, limit, _metric) in BENCHMARKS.items():
    print(f"{name:15s} task={task} fewshot={shots} limit/subtask={limit or 'all'}")

## 1. Hàm chạy lm-eval

In [ ]:
def run_lm_eval(label: str, task: str, shots: int, limit: int | None) -> dict:
    model_args = f"pretrained={C.SFT_MERGED},dtype={DTYPE},enable_thinking=False"
    if label == "dpo":
        model_args += f",peft={C.DPO_ADAPTER}"
    out_dir = C.EVAL_DIR / "lm_eval" / f"{label}-{task}"
    cmd = [
        "lm_eval", "--model", "hf", "--model_args", model_args,
        "--tasks", task, "--num_fewshot", str(shots),
        "--apply_chat_template", "--batch_size", BATCH,
        "--device", "cuda:0", "--seed", str(C.SEED), "--output_path", str(out_dir),
    ]
    if shots:
        cmd.append("--fewshot_as_multiturn")
    if limit:
        cmd += ["--limit", str(limit)]
    print(f"\n>>> {label} · {task}\n{' '.join(cmd)}")
    proc = subprocess.run(cmd, capture_output=True, text=True)
    files = sorted(out_dir.glob("**/results*.json"), key=lambda p: p.stat().st_mtime)
    if proc.returncode != 0 or not files:
        print(proc.stderr[-2000:])
        raise RuntimeError(f"lm_eval failed for {label}/{task}")
    return json.loads(files[-1].read_text())


def score(result: dict, task: str, metric: str) -> tuple[float, float]:
    # Groups (global_mmlu_full_vi) report the aggregate under the group name.
    block = result.get("groups", {}).get(task) or result["results"][task]
    stderr_key = metric.replace(",", "_stderr,", 1)
    return float(block[metric]), float(block.get(stderr_key, float("nan")))

## 2. Chạy (T4: ~40–60 phút cho cả hai điều kiện)

In [ ]:
rows = []
for name, (task, shots, limit, metric) in BENCHMARKS.items():
    row = {"benchmark": name, "task": task, "limit_per_subtask": limit}
    for label in ("sft", "dpo"):
        value, err = score(run_lm_eval(label, task, shots, limit), task, metric)
        row[label], row[f"{label}_stderr"] = value, err
    row["delta"] = row["dpo"] - row["sft"]
    rows.append(row)
    print(row)

## 3. Biểu đồ (sản phẩm nộp `07-benchmark-comparison.png`)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.DataFrame(rows)
print(df.to_string(index=False))

x = np.arange(len(df))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - 0.18, df["sft"], 0.36, yerr=df["sft_stderr"], label="SFT", color="#2e548a", capsize=3)
ax.bar(x + 0.18, df["dpo"], 0.36, yerr=df["dpo_stderr"], label="SFT+DPO", color="#c83538", capsize=3)
for i, r in df.iterrows():
    ax.annotate(f"Δ={r['delta']:+.3f}", (x[i], max(r["sft"], r["dpo"]) + 0.05), ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(df["benchmark"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("score (± stderr)")
ax.set_title(f"lm-eval · chat template · {C.COMPUTE_TIER}")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig.savefig(C.SCREENSHOTS / "07-benchmark-comparison.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
(C.EVAL_DIR / "benchmark_results.json").write_text(
    json.dumps({"compute_tier": C.COMPUTE_TIER, "dtype": DTYPE, "chat_template": True, "results": rows}, indent=2)
)
print("Saved data/eval/benchmark_results.json")

## 4. Đọc kết quả (REFLECTION §7)

1. |Δ| có lớn hơn ~2× stderr không? Nếu không, đừng gọi là "tăng" hay "giảm".
2. IFEval đo khả năng làm đúng định dạng: DPO trên dữ liệu chat thường giúp hoặc giữ nguyên.
3. GSM8K giảm ⇒ alignment tax (thuế căn chỉnh); NB7 (GRPO với reward kiểm chứng được) là một cách lấy lại.
4. Global-MMLU-vi gần như phẳng là bình thường: DPO không dạy kiến thức mới.

**Tiếp theo:** NB7 (GRPO, thưởng điểm) hoặc `make verify`.

In [ ]:
backup_bonus('06_benchmark')

In [ ]:
# Free GPU memory held by the previous stage (model, trainer, llama.cpp handle).
import gc
for _name in ('trainer', 'model', 'ref_model', 'llm', 'policy', 'ref', 'tokenizer'):
    globals().pop(_name, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
except ImportError:
    pass

# NB7 — GRPO với reward kiểm chứng được (RLVR) (THƯỞNG, +8)

DPO học từ *cặp* sở thích có sẵn (ngoại tuyến). GRPO (DeepSeekMath, 2024; dùng trong
DeepSeek-R1) sinh **G câu trả lời cho mỗi câu hỏi**, chấm bằng hàm reward, rồi đẩy
xác suất các câu có reward cao hơn trung bình nhóm. Không cần mô hình reward hay
mô hình critic: với toán, reward là "đáp số đúng hay sai" (RLVR, *verifiable rewards*).

Notebook này chạy một vòng GRPO rất nhỏ để thấy cơ chế, **không** để đạt điểm cao.
T4: ~40 phút cho 60 bước với G=4.

**Dữ liệu:** `vuongtsc/vi-gsm8k-agentic` (MIT): 1.465 bài toán tiểu học viết mới bằng
tiếng Việt từ seed GSM8K, đáp số dạng số đã kiểm tra bằng chạy code. Bộ này chỉ có
split `train`, nên notebook tự chia cố định thành tập huấn luyện/kiểm tra. Các bài được lọc để mô hình yếu
giải sai, nên độ chính xác ban đầu của mô hình 4B có thể thấp: nếu cả G câu trả lời của một
câu hỏi đều sai thì advantage = 0 và câu hỏi đó không đóng góp gradient.

| | DPO (NB3) | GRPO (NB7) |
|---|---|---|
| Dữ liệu | cặp chosen/rejected cố định | chỉ cần câu hỏi + cách chấm |
| Sinh trong lúc huấn luyện | không | có (tự sinh, on-policy) |
| Reference / KL | bắt buộc (β) | tuỳ chọn; TRL mặc định `beta=0.0` |
| Chi phí | 2 forward / cặp | G lần sinh / câu hỏi |

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab22" / "config.py").exists())
sys.path.insert(0, str(ROOT))

import unsloth  # noqa: F401
import torch

from lab22 import config as C
from lab22 import math_reward as MR
from lab22 import modeling as MD

assert torch.cuda.is_available()
assert C.SFT_MERGED.exists(), "Run NB1 first"
C.ensure_dirs()

BIG = C.COMPUTE_TIER == "BIGGPU"
N_TRAIN, N_TEST, MAX_STEPS, G = (1265, 200, 200, 8) if BIG else (400, 100, 60, 4)

## 1. Dữ liệu + hàm reward

In [ ]:
from datasets import load_dataset

INSTRUCTION = "Giải bài toán sau. Suy luận ngắn gọn, rồi kết thúc bằng dòng 'Đáp số: <số>'.\n\n"


def gold(answer: str) -> str:
    return answer.split("####")[-1].strip().replace(",", "")


def to_row(r):
    # vi-gsm8k-agentic stores the number in `final_answer`; GSM8K puts it after "####".
    ref = str(r["final_answer"]) if "final_answer" in r else gold(r["answer"])
    return {"prompt": [{"role": "user", "content": INSTRUCTION + r["question"]}], "answer": ref.strip()}


if C.GRPO_DATASET == "openai/gsm8k":
    gsm = load_dataset("openai/gsm8k", "main")
    train_raw, test_raw = gsm["train"].shuffle(seed=C.SEED), gsm["test"]
else:
    # Single split: hold out a fixed test slice before any training row is drawn.
    parts = load_dataset(C.GRPO_DATASET, split="train").shuffle(seed=C.SEED).train_test_split(
        test_size=N_TEST, seed=C.SEED
    )
    train_raw, test_raw = parts["train"], parts["test"]
cols = train_raw.column_names
train_ds = train_raw.select(range(min(N_TRAIN, len(train_raw)))).map(to_row, remove_columns=cols)
test_ds = test_raw.select(range(min(N_TEST, len(test_raw)))).map(to_row, remove_columns=cols)
assert not set(train_ds["prompt"][i][0]["content"] for i in range(len(train_ds))) & set(
    test_ds["prompt"][i][0]["content"] for i in range(len(test_ds))
), "GRPO train/test overlap"
print(f"{C.GRPO_DATASET}: train {len(train_ds)} · test {len(test_ds)}")

# Đọc số theo cả kiểu Việt ("1.440", "2,5") lẫn kiểu Anh ("1,440", "2.5"): xem lab22/math_reward.py.
correctness_reward, format_reward = MR.correctness_reward, MR.format_reward
assert MR.is_correct(MR.extract_answer("... vậy\nĐáp số: 1.250"), "1250")
assert MR.is_correct(MR.extract_answer("Đáp số: 2,5 kg"), "2.5")
assert correctness_reward([[{"content": "Đáp số: 7"}]], ["8"]) == [0.0]

## 2. Độ chính xác trước khi huấn luyện

In [ ]:
def accuracy(model, tokenizer) -> float:
    prompts = [r["prompt"][0]["content"] for r in test_ds]
    outs = MD.generate(model, tokenizer, prompts, max_new_tokens=320)
    return sum(MR.is_correct(MR.extract_answer(o), r["answer"]) for o, r in zip(outs, test_ds)) / len(test_ds)


model, tokenizer = MD.load_model(C.SFT_MERGED)
acc_before = accuracy(model, tokenizer)
print(f"test[{len(test_ds)}] accuracy before GRPO: {acc_before:.3f}")
model = MD.add_lora(model)

## 3. GRPO

`per_device_train_batch_size × grad_accum` phải chia hết cho `num_generations` (G):
mỗi batch chứa trọn các nhóm G câu trả lời của cùng một câu hỏi.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

args = GRPOConfig(
    output_dir=str(C.ADAPTERS / "grpo-checkpoints"),
    per_device_train_batch_size=G,
    gradient_accumulation_steps=1,
    num_generations=G,
    max_completion_length=320,
    max_steps=MAX_STEPS,
    learning_rate=5e-6,
    warmup_steps=0.1,
    lr_scheduler_type="cosine",
    temperature=1.0,
    chat_template_kwargs=C.CHAT_TEMPLATE_KWARGS,
    logging_steps=5,
    save_strategy="no",
    optim="adamw_8bit",
    seed=C.SEED,
    report_to="none",
    **MD.precision_flags(),
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward, format_reward],
    args=args,
    train_dataset=train_ds,
    processing_class=tokenizer,
)
trainer.train()

## 4. Đường cong reward + độ chính xác sau huấn luyện (sản phẩm nộp `08-grpo-reward.png`)

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

logs = pd.DataFrame([r for r in trainer.state.log_history if "reward" in r])
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(logs["step"], logs["reward"], color="#2e548a", label="mean reward")
if "reward_std" in logs:
    ax.fill_between(logs["step"], logs["reward"] - logs["reward_std"], logs["reward"] + logs["reward_std"], alpha=0.2)
ax.set_xlabel("step")
ax.set_ylabel("reward (max 2.5)")
ax.set_title(f"GRPO · {C.GRPO_DATASET.split('/')[-1]} · G={G}")
ax.grid(True, alpha=0.3)
fig.savefig(C.SCREENSHOTS / "08-grpo-reward.png", dpi=120, bbox_inches="tight")
plt.show()

acc_after = accuracy(trainer.model, tokenizer)
trainer.model.save_pretrained(str(C.GRPO_ADAPTER))
result = {
    "dataset": C.GRPO_DATASET, "n_test": len(test_ds), "steps": MAX_STEPS,
    "num_generations": G, "acc_before": acc_before, "acc_after": acc_after,
}
(C.GRPO_ADAPTER / "grpo_metrics.json").write_text(json.dumps(result, indent=2))
print(result)

## 5. Câu hỏi

1. Reward tăng nhanh nhất ở thành phần nào: định dạng hay tính đúng? Đó có phải "khai thác lỗ hổng reward (reward hacking)" không?
2. Với N_TEST=100, chênh lệch độ chính xác bao nhiêu mới vượt nhiễu? (sai số chuẩn ≈ √(p(1−p)/n)).
3. So `acc_before` với GSM8K (tiếng Anh) của SFT ở NB6: ngôn ngữ, độ khó, câu hỏi và cách chấm
   đều khác, nên con số nào đáng tin hơn cho câu hỏi "GRPO có giúp toán không"?

In [ ]:
backup_bonus('07_grpo_bonus')

In [ ]:
# Free GPU memory held by the previous stage (model, trainer, llama.cpp handle).
import gc
for _name in ('trainer', 'model', 'ref_model', 'llm', 'policy', 'ref', 'tokenizer'):
    globals().pop(_name, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
except ImportError:
    pass

## Beta-sweep — ba lượt cùng split và ngân sách

In [ ]:
for beta, label in [(0.05, '0.05'), (0.1, '0.10'), (0.5, '0.50')]:
    subprocess.run([sys.executable, 'scripts/train_dpo.py', '--beta', str(beta), '--output-dir', f'adapters/dpo-b{label}'], check=True)
    backup_bonus('beta'+label)
subprocess.run([sys.executable, 'scripts/eval_judge.py', '--plot-sweep'], check=True)
backup_bonus('complete')

Sau mỗi mục, lưu phiên bản có output. ZIP bằng chứng không gồm trọng số; tải adapter/GGUF riêng nếu cần. Chỉ cập nhật REFLECTION từ số liệu thực tế.